# Dynamic ACP Capability Evidence and Upstream Alignment

**Status:** Approved for implementation on 2026-09-01.  
**Design epic:** `bd-j5kb`  
**Prior live audit:** `bd-3qrp`  
**Authoritative artifact:** this notebook.

This design removes the chicken-and-egg dependency in which Spur can only recognize capabilities already represented by its typed/provider-specific implementation. It is grounded in the Grok 1.0.13 and Kiro 2.20.2 live probes; Claude Code remains an authenticated-live-evidence gap because its probe failed before protocol exchange.

## Formal profile pins

| Profile | Version | Registry state | Purpose |
|---|---:|---|---|
| `relational_lia` | 1 | implemented | Dispatch partition, determinism, coverage, and reachability |
| `state_invariant_lia` | 1 | implemented | Capability-confidence initiation and transition safety |

The architecture choice was independently evaluated with Z3 Optimize. Dynamicity-first lexicographic optimization selected `hybrid_isolated` (`sol_c9bbbc922f344a7a`); the complete Pareto frontier contains that design and `manifest_only` (`sol_91cbde4035ae45e6`). Cost-first sensitivity selects `manifest_only` (`sol_d99bfab321cf4fdb`), making the chosen result explicitly conditional on automatic alignment being preferred after mandatory safety.

## Decision

Adopt a provider-neutral, isolated hybrid evidence kernel:

`raw upstream frames → evidence ledger → semantic capabilities → policy reducer → exactly one dispatch`

The raw frame is captured before typed ACP projection. Standard ACP fields, vendor fields, accepted probe results, notifications, and failures become evidence records with provenance rather than provider-specific truth tables. A reducer derives an immutable capability snapshot for one CLI identity and evidence epoch.

An unseen or changed CLI identity starts from passive evidence and safe fallback behavior. A single ephemeral background process may run non-billed probe recipes without mutating the user's ACP session. Results are committed atomically to the cache and become eligible at the next dispatch boundary.

A probe recipe is only a way to ask an upstream question. Its existence never advertises a command, supplies model/effort allowlists, or proves native support.

### Goals

- Preserve unknown upstream data before schema projection.
- Learn model, effort, mode, and command identifiers from upstream evidence.
- Keep normalization provider-neutral while retaining provenance.
- Fail closed for unverified native dispatch while retaining safe advertised prompt commands.
- Select exactly one route per action and never retry through a second route after a partial native attempt.
- Invalidate evidence when executable identity changes.
- Generate replay fixtures from probe output for deterministic regression tests.

### Non-goals

- Sending billed prompts to discover capabilities.
- Fuzzing arbitrary methods or parameter shapes.
- Treating authentication failure, timeout, or malformed data as proof of unsupported behavior.
- Reworking unrelated ACP sequencing or the entire TUI command system.
- Removing legacy provider adapters before shadow parity is demonstrated.

In [ ]:
flowchart TD
    SPEC["`@spec ACP-CAPABILITY-ROUTER
@type Route = enum[hidden, prompt_only, native_preferred]
@input prompt_advertised: Bool
@input native_verified: Bool
@input identity_matches: Bool
@input native_failed: Bool
@output status: Route
@requires INPUT_DOMAIN: true`"]

    NATIVE["`@branch NATIVE
@when native_verified and identity_matches and not native_failed
@ensures NATIVE_ROUTE: status = native_preferred`"]

    PROMPT["`@branch PROMPT
@when prompt_advertised and not (native_verified and identity_matches and not native_failed)
@ensures PROMPT_ROUTE: status = prompt_only`"]

    HIDDEN["`@branch HIDDEN
@when not prompt_advertised and not (native_verified and identity_matches and not native_failed)
@ensures HIDDEN_ROUTE: status = hidden`"]

    CHECK["`@verify CONSISTENT: witness consistency
@verify DETERMINISTIC: prove determinism
@verify COVERED: prove partition_coverage
@verify EXCLUSIVE: prove partition_exclusive
@verify ALL_STATUSES: witness each status`"]

    SPEC --> NATIVE --> CHECK
    SPEC --> PROMPT --> CHECK
    SPEC --> HIDDEN --> CHECK

## Evidence model and ownership

| Contract | Required fields | Ownership |
|---|---|---|
| `CliIdentity` | resolved executable, upstream version, argv fingerprint, non-secret environment fingerprint | Probe/cache boundary |
| `CapabilityKey` | semantic kind plus upstream-stable identifier | Evidence kernel |
| `EvidenceRecord` | capability key, claim, provenance, identity, observation time, raw digest, session scope | Append-only evidence ledger |
| `ReducedCapability` | confidence status, selected route, source summary, evidence epoch | Policy reducer |
| `ProbeRecipe` | method, parameter derivation, safety class, timeout, response interpretation | Small declarative registry |

Provenance distinguishes standard advertisement, vendor advertisement, accepted active probe, rejected active probe, observed notification, and prompt fallback. Raw payloads remain available by digest even when current typed schemas omit fields such as top-level `models`.

Choice values—including model IDs, effort IDs, mode IDs, and command names—come from observed upstream payloads. Recipes may name method shapes that cannot otherwise be discovered, but they must not embed the valid choice set.

A recipe-only candidate is not display-eligible. Safe prompt routing requires passive upstream advertisement; native routing requires standard contract evidence or an accepted isolated probe bound to the current CLI identity.

In [ ]:
stateDiagram-v2
    [*] --> Hidden
    Hidden --> PromptOnly: candidate_observed
    PromptOnly --> NativePreferred: native_verified
    NativePreferred --> PromptOnly: invalidate

    note right of Hidden
      @spec ACP-CAPABILITY-EVIDENCE-LIFECYCLE
      @type EvidenceLevel = enum[hidden, prompt_only, native_preferred]
      @type EvidenceEvent = enum[candidate_observed, native_verified, invalidate]
      @input event: EvidenceEvent
      @state-var level: EvidenceLevel
      @state-var native_allowed: Bool
      @requires INITIAL: level = hidden and native_allowed = false
      @state Hidden
      @invariant SAFE_NATIVE: native_allowed = (level = native_preferred)
    end note

    note right of PromptOnly
      @state PromptOnly
      @transition OBSERVE
      @from Hidden
      @to PromptOnly
      @event event = candidate_observed
      @guard level = hidden and native_allowed = false
      @update level' = prompt_only
      @update native_allowed' = false
    end note

    note right of NativePreferred
      @state NativePreferred
      @transition PROMOTE
      @from PromptOnly
      @to NativePreferred
      @event event = native_verified
      @guard level = prompt_only and native_allowed = false
      @update level' = native_preferred
      @update native_allowed' = true
    end note

    note left of PromptOnly
      @transition DEMOTE
      @from NativePreferred
      @to PromptOnly
      @event event = invalidate
      @guard level = native_preferred and native_allowed = true
      @update level' = prompt_only
      @update native_allowed' = false
      @verify INIT_SAFE: prove initiate SAFE_NATIVE
      @verify OBSERVE_SAFE: prove preserve SAFE_NATIVE on OBSERVE
      @verify PROMOTE_SAFE: prove preserve SAFE_NATIVE on PROMOTE
      @verify DEMOTE_SAFE: prove preserve SAFE_NATIVE on DEMOTE
    end note

## Runtime protocol and failure handling

1. Capture initialization and new-session frames before typed deserialization.
2. Normalize passive evidence and publish an immediate safe snapshot.
3. On a cache miss or CLI identity change, coalesce requests into one isolated, ephemeral, non-billed probe.
4. Merge probe evidence into a new immutable epoch; never mutate the user's ACP session.
5. Each command dispatch captures one epoch and one route. Registry refresh may adopt a later epoch only between dispatches.
6. A native failure records invalidation for subsequent actions. The original action is never automatically resent as prompt text.

| Observation | Evidence effect | User-visible behavior |
|---|---|---|
| Standard ACP advertisement | Native evidence | Native-preferred when identity matches |
| Vendor command advertisement only | Prompt evidence | Safe prompt route |
| Isolated probe accepted | Native evidence | Promote at an epoch boundary |
| Method not found / explicit rejection | Rejected evidence | Keep prompt route if advertised |
| Authentication required | Inconclusive | Preserve existing safe routes; request authentication separately |
| Timeout / transport close | Unknown | Preserve existing safe routes; retry only under bounded policy |
| CLI identity drift | Invalidate native evidence | Demote until reprobed |
| Native dispatch fails | Invalidate after action | No same-action fallback; next action uses reduced route |

Probe output and cache records must redact secrets. Raw digests are stable; raw payload retention follows the existing artifact policy.

## Integration seams and migration

| Seam | Designed change |
|---|---|
| `crates/spur-acp/src/connection/native.rs` | Capture raw initialization/session envelopes before schema projection and pass them to normalization. |
| `crates/spur-acp/src/spur_agent_caps.rs` | Become a compatibility facade over reduced evidence instead of owning Grok/Kiro truth snapshots. |
| `crates/spur-acp/src/agent_model_catalog.rs` | Evolve the identity-keyed model cache into a versioned evidence cache while preserving TTL behavior. |
| `crates/spur-tui/src/commands/advertised.rs` | Consume one reduced route per capability and deduplicate dynamic/synthetic collisions before registry insertion. |
| `scripts/probe_acp_capabilities.py` | Emit machine-readable raw observations, normalized claims, identity, and fixture artifacts without assuming current Spur support. |
| ACP/TUI integration tests | Replay generated fixtures and assert parity, invalidation, uniqueness, and safe fallback. |

Migration is staged:

1. Freeze probe-output and fixture schemas.
2. Add raw capture, evidence types, and reducer in shadow mode with no routing change.
3. Add isolated probing and identity-keyed cache.
4. Switch advertised model/effort/command construction to reduced evidence while retaining a bounded legacy fallback.
5. Remove provider-specific snapshots only after Grok/Kiro replay parity and authenticated Claude coverage are demonstrated.

Every stage is independently revertible. Cache schema changes use an explicit version; incompatible entries are ignored, not partially interpreted.

## TDD, evaluation, and task boundaries

Every implementation task follows the same pre/post loop:

1. **Pre-evaluation:** run the current probe or fixture replay and preserve the failing/misaligned observation.
2. **Red:** add the narrow unit, round-trip, collision, or integration test that expresses the approved contract.
3. **Green:** make the smallest crate-local change.
4. **Post-evaluation:** rerun the same probe/fixture command and compare its machine-readable result.
5. **Regression:** run the affected crate tests through `scripts/spur-cargo`; never bare `cargo`.

Required scenarios include:

- Top-level or unknown vendor fields survive typed-schema omissions.
- New upstream model/effort IDs require no source allowlist change.
- Recipe existence alone cannot advertise or native-enable a capability.
- Auth failure and timeout remain inconclusive.
- CLI identity change demotes previously verified native support.
- Dynamic and synthetic command collisions produce one registry entry and one dispatch.
- A failed native action is not resent through prompt fallback.
- Generated Grok/Kiro fixtures replay deterministically; Claude joins once authenticated.

Plausible implementation boundaries are: probe/fixture contract, ACP evidence kernel, reducer/lifecycle tests, isolated runner/cache, TUI registry integration, and live replay/evaluation. The implementation plan must order these by contract dependency and keep file scopes non-overlapping where possible.

## Acceptance criteria

The work is ready to replace the legacy path only when all formal cells are fresh and green, fixture replay passes, Grok and Kiro post-probes agree with the reducer, and the legacy-vs-evidence shadow comparison has no unexplained route differences. Claude Code absence is reported as an evidence gap, never silently treated as unsupported.

## Proof manifest

| Formal spec | Cell ID | Profile | Source hash | IR hash | Obligation-set hash | Report hash | Obligations | Status |
|---|---|---|---|---|---|---|---:|---|
| `ACP-CAPABILITY-ROUTER` | `bd350001-0000-4000-8000-000000000003` | `relational_lia@1` | `5da03aeae487437674f3d979f0e9555cf7f7943bc455aca0ca845380d64f4050` | `72760e46a69669d1f525ebc1876f00fb469ce8f8f57f3ecf08eecbccfb5eae43` | `b73840459fa2ef8a7a7dd5b6d5256e83b8e2bdf64aa9ff8eb4d1e62ff25daff7` | `63631bebb75ffa69cdeecc8b3ae686d5a91396dfab030beb7d536842389c338d` | 7/7 | solver-verified, fresh |
| `ACP-CAPABILITY-EVIDENCE-LIFECYCLE` | `bd350001-0000-4000-8000-000000000005` | `state_invariant_lia@1` | `854c4d4340cd7ecad11bcf60f9501309cd99f077988321cff8636605214e83af` | `fd06eff6e75d589567cce709a309da6ca751c06864851fa4de4899638af37d96` | `ffffc2ae588dfa12e99b5b76f7c4e66f3805616823e7cb3fce74ea8c6ad4d6cc` | `e81e519f56896645475eae4af1344e437575e7b9941fbc29100a89d06361d923` | 8/8 | solver-verified, fresh |

The executed `application/vnd.spur.ns-proof+json` bundles on the formal cells are authoritative. Each bundle reports `formal_status=verified`, `solver_verified=true`, `proof_fresh=true`, and matched source, metadata, and data freshness. Independent architecture Optimize evidence is listed in the opening cell and the design epic.